# Nouvelle section Partie Contacts

In [ ]:
# installation des programmes
!pip install unidecode
!pip install selenium
!pip install ydata_profiling
!pip install rapidfuzz

In [ ]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver import ChromeOptions
from datetime import datetime
from unidecode import unidecode
from ast import literal_eval
from itertools import chain
from ydata_profiling import ProfileReport
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
import unicodedata
import re
import numpy as np
import pandas as pd

In [ ]:
#lire le fichier csv CONTACT ODOO
df_Contacts = pd.read_excel('/content/Contact_Particulier2.xlsx')
#Afficher tout le df
pd.set_option('display.max_rows', None)

In [ ]:
#Verifier si le Nom est ecrit sans contact en plus
df_Contacts[df_Contacts["Nom affiché"].str.contains("-", na=False)]

In [ ]:
#Verifier si le mail est ecrit correctement
mask = df_Contacts["Courriel"].astype(str).str.contains(r"<.*?>", regex=True, na=False)
df_Contacts[mask]

In [ ]:
import html

col = "Courriel"
s = df_Contacts[col].map(lambda x: html.unescape(x) if isinstance(x, str) else x)

pattern = r"<\s*([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})\s*>"
email = s.str.extract(pattern, expand=False)

df_Contacts[col] = email.fillna(s)

In [ ]:
#supprimer les lignes contenant des mails
df_Contacts2 = df_Contacts[~df_Contacts['Nom affiché'].str.contains(r'@', na=False)]

#supprimer les lignes ne contenant pas de noms
df_Contacts3 = df_Contacts2.dropna(subset=["Contact"])

#supprime les lignes ne contenant ni telephone, ni mail, ni mobile
df_Contacts4 = df_Contacts3.dropna(subset=["Courriel", "Téléphone",'Mobile'], how="all")

In [ ]:
#retire les espaces invisibles
cols = ["Nom affiché", "Contact", "Poste Occupé", "Courriel", "Titre"]

df_Contacts4[cols] = df_Contacts4[cols].apply(lambda s: s.astype(str).str.strip())


In [ ]:
#remet les nan en np.nan
cols = ["Nom affiché", "Contact", "Poste Occupé", "Courriel", "Titre"]

df_Contacts4[cols] = df_Contacts4[cols].replace(
    to_replace=[r"^\s*nan\s*$", r"^\s*NaN\s*$", r"^\s*None\s*$", r"^\s*<NA>\s*$"],
    value=np.nan,
    regex=True
)

In [ ]:
output_file = "df_Contacts4.xlsx"
df_Contacts4.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)